<a href="https://colab.research.google.com/github/Anggasaputraramadhan/PCVK/blob/main/week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import math
from google.colab.patches import cv2_imshow

def show_image(img, title='Image'):
    plt.figure(figsize=(6, 6))
    if len(img.shape) == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
path = '/content/drive/MyDrive/KULIAH/PCVK/Images/peppers.jpg'
img = cv2.imread(path)

if img is None:
    print(f"Error: Gagal memuat citra dari path: {path}. Pastikan path dan nama file sudah benar.")
else:
    inverse_img = 255 - img

    final_frame = cv2.hconcat((img, inverse_img))
    cv2_imshow(final_frame)

In [ ]:
path = '/content/drive/MyDrive/KULIAH/PCVK/Images/peppers.jpg'
img = cv2.imread(path)

if img is None:
    print(f"Error: Could not load image from path: {path}")
else:
    inverse_img = 255 - img

    final_frame = cv2.hconcat((img, inverse_img))
    cv2_imshow(final_frame)

In [ ]:
path = '/content/drive/MyDrive/KULIAH/PCVK/Images/houses.jpg'
img = cv2.imread(path)

c_val = int(input('Masukkan nilai kecerahan (konstanta c): '))

img_float = img.astype(np.float32)
log_img = c_val * np.log(1 + img_float)

log_img = cv2.normalize(log_img, None, 0, 255, cv2.NORM_MINMAX)
log_img = np.uint8(log_img)

final_frame = cv2.hconcat((img, log_img))
cv2_imshow(final_frame)

In [ ]:
path = '/content/drive/MyDrive/KULIAH/PCVK/Images/peppers.jpg'
img = cv2.imread(path)
B, G, R = cv2.split(img.astype(np.float32))

gray_avg = np.uint8((R + G + B) / 3)

gray_lightness = np.uint8((np.maximum(np.maximum(R, G), B) + np.minimum(np.minimum(R, G), B)) / 2)

gray_luminance = np.uint8(0.21 * R + 0.72 * G + 0.07 * B)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1), plt.imshow(gray_avg, cmap='gray'), plt.title('Averaging'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(gray_lightness, cmap='gray'), plt.title('Lightness'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(gray_luminance, cmap='gray'), plt.title('Luminance'), plt.axis('off')
plt.show()

In [ ]:
path = '/content/drive/MyDrive/KULIAH/PCVK/Images/peppers.jpg'
img = cv2.imread(path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

lower_red1 = np.array([0, 50, 50])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 50, 50])
upper_red2 = np.array([180, 255, 255])

mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
mask = mask1 | mask2

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

result = np.where(mask[:, :, None] == 255, img_rgb, gray_3ch)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1), plt.imshow(img_rgb), plt.title('Citra Asli'), plt.axis('off')
plt.subplot(1, 2, 2), plt.imshow(result), plt.title('Filter Warna Merah + Grayscale Lainnya'), plt.axis('off')
plt.show()

In [ ]:
path = '/content/drive/MyDrive/KULIAH/PCVK/Images/houses.jpg'
img = cv2.imread(path)

gamma = float(input('Masukkan nilai Gamma: '))

inv_gamma = 1.0 / gamma
gamma_img = np.array(255 * (img / 255.0) ** inv_gamma, dtype=np.uint8)

final_frame = cv2.hconcat((img, gamma_img))
cv2_imshow(final_frame)

In [ ]:
bit_depth = int(input('Masukkan bit depth tujuan (1-8): '))
level = 255 / (pow(2, bit_depth) - 1)

original = cv2.imread('/content/drive/MyDrive/KULIAH/PCVK/Images/peppers.jpg', cv2.IMREAD_GRAYSCALE)

depth_image = np.round(original / level) * level
depth_image = np.clip(depth_image, 0, 255).astype(np.uint8)

final_frame = cv2.hconcat((original, depth_image))
cv2_imshow(final_frame)
print(f"Bit depth awal: 8 bit")
print(f"Bit depth tujuan: {bit_depth} bit")
print(f"Jumlah level: {int(pow(2, bit_depth))}")

In [ ]:
original_img = cv2.imread('/content/drive/MyDrive/KULIAH/PCVK/Images/galaxy.jpg')

cv_img = []
for img_path in glob.glob('/content/drive/MyDrive/KULIAH/PCVK/Images/noises/*.jpg'):
    n = cv2.imread(img_path)
    if n is not None:
        cv_img.append(n.astype(np.float32))

def calculate_psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    max_pixel = 255.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

jumlah_list = [10, 20, 40, 80, 100]
for jml in jumlah_list:
    if len(cv_img) >= jml:
        subset = cv_img[:jml]
        avg_denoised = np.mean(subset, axis=0).astype(np.uint8)
        psnr_val = calculate_psnr(original_img, avg_denoised)
        print(f"Jumlah Citra di-Average: {jml:3d} | Nilai PSNR: {psnr_val:.2f} dB")

        if jml in [10, 100]:
            show_image(avg_denoised, f"Denoised Average {jml} Citra (PSNR: {psnr_val:.2f}dB)")

In [ ]:
imgA = cv2.imread('/content/drive/MyDrive/KULIAH/PCVK/Images/couple.tiff')
h, w, _ = imgA.shape
maskB = np.zeros((h, w, 3), dtype=np.uint8)
cv2.circle(maskB, (w // 3, h // 3), 80, (255, 255, 255), -1)
cv2.circle(maskB, (2 * w // 3, h // 3), 80, (255, 255, 255), -1)

res_not = cv2.bitwise_not(imgA)
res_or = cv2.bitwise_or(imgA, maskB)
res_and = cv2.bitwise_and(imgA, maskB)
res_nand = cv2.bitwise_not(res_and)
res_xor = cv2.bitwise_xor(imgA, maskB)

show_image(res_and, "Image Masking - Operator AND")

In [ ]:
path_malam = '/content/drive/MyDrive/KULIAH/PCVK/Images/night.jpg'
img_night = cv2.imread(path_malam)

if img_night is not None:
    gamma_val = 0.4
    inv_gamma = 1.0 / gamma_val
    night_corrected = np.array(255 * (img_night / 255.0) ** inv_gamma, dtype=np.uint8)

    final_frame = cv2.hconcat((img_night, night_corrected))
    cv2_imshow(final_frame)

In [ ]:
path_cray = '/content/drive/MyDrive/KULIAH/PCVK/Images/crayfish.jpg'
cray_img = cv2.imread(path_cray)

if cray_img is not None:
    contrast_val = 1.3
    factor_c = (259 * (contrast_val + 255)) / (255 * (259 - contrast_val))
    temp_img = factor_c * (cray_img.astype(np.float32) - 128) + 128
    temp_img = np.clip(temp_img, 0, 255).astype(np.uint8)

    gamma_v = 0.6
    inv_g = 1.0 / gamma_v
    final_cray = np.array(255 * (temp_img / 255.0) ** inv_g, dtype=np.uint8)

    final_frame = cv2.hconcat((cray_img, final_cray))
    cv2_imshow(final_frame)